# Exercice 09 : Répondre à des questions

À partir du fichier nettoyé à l'exercice 08.

In [8]:
import pandas as pd

titanic = pd.read_csv("../exercice-08/data/titanic_propre.csv")
print(titanic.shape)
print(titanic.head())

(889, 9)
   survived     sex   age  sibsp  parch     fare  class  embark_town  alone
0     False    male  22.0      1      0   7.2500  Third  Southampton  False
1      True  female  38.0      1      0  71.2833  First    Cherbourg  False
2      True  female  26.0      0      0   7.9250  Third  Southampton   True
3      True  female  35.0      1      0  53.1000  First  Southampton  False
4     False    male  35.0      0      0   8.0500  Third  Southampton   True


## 1. Les femmes ont-elles mieux survécu que les hommes dans chaque classe ?

In [9]:
survie_classe_sexe = pd.pivot_table(
    titanic, values="survived",
    index="class", columns="sex", aggfunc="mean",
).round(2)
print(survie_classe_sexe)

sex     female  male
class               
First     0.97  0.37
Second    0.92  0.16
Third     0.50  0.14


Oui, dans les trois classes : 97 % des femmes de 1re ont survécu contre 37 % des hommes, 92 % contre 16 % en 2e, et 50 % contre 14 % en 3e. L'écart le plus fort est en 2e classe.

## 2. Les enfants ont-ils été prioritaires ?

In [10]:
titanic["tranche_age"] = pd.cut(
    titanic["age"], bins=[0, 12, 18, 60, 100],
    labels=["enfant", "ado", "adulte", "senior"],
)
survie_par_age = titanic.groupby("tranche_age").agg(
    passagers=("survived", "size"),
    taux_survie=("survived", "mean"),
).round(2)
print(survie_par_age)

             passagers  taux_survie
tranche_age                        
enfant              69         0.58
ado                 70         0.43
adulte             729         0.36
senior              21         0.19


Oui : 58 % des enfants (12 ans et moins) ont survécu, contre 43 % des ados, 36 % des adultes et 19 % des seniors. Plus les passagers sont jeunes, mieux ils ont survécu.

## 3. Quel port a le prix moyen le plus élevé ? La répartition des classes l'explique-t-elle ?

In [11]:
prix_moyen_par_port = titanic.groupby("embark_town")["fare"].mean().sort_values(ascending=False).round(2)
print(prix_moyen_par_port)

embark_town
Cherbourg      59.95
Southampton    27.08
Queenstown     13.28
Name: fare, dtype: float64


In [12]:
classes_par_port = pd.crosstab(titanic["embark_town"], titanic["class"], normalize="index").round(2)
print(classes_par_port)

class        First  Second  Third
embark_town                      
Cherbourg     0.51    0.10   0.39
Queenstown    0.03    0.04   0.94
Southampton   0.20    0.25   0.55


In [13]:
prix_par_port_et_classe = pd.pivot_table(
    titanic, values="fare",
    index="embark_town", columns="class", aggfunc="mean",
).round(2)
print(prix_par_port_et_classe)

class         First  Second  Third
embark_town                       
Cherbourg    104.72   25.36  11.21
Queenstown    90.00   12.35  11.18
Southampton   70.36   20.33  14.64


Cherbourg a le prix moyen le plus élevé (60 £, contre 27 £ à Southampton et 13 £ à Queenstown). La répartition des classes l'explique en grande partie : la moitié des passagers de Cherbourg voyagent en 1re classe, contre 20 % à Southampton et 3 % à Queenstown. Mais pas entièrement : à classe égale, la 1re coûte aussi plus cher au départ de Cherbourg (105 £ contre 70 £ à Southampton).

## 4. Voyager seul augmentait-il ou diminuait-il les chances de survie ?

In [14]:
survie_seul = titanic.groupby("alone").agg(
    passagers=("survived", "size"),
    taux_survie=("survived", "mean"),
).round(2)
print(survie_seul)

       passagers  taux_survie
alone                        
False        354         0.51
True         535         0.30


Voyager seul diminuait les chances de survie : 30 % des passagers seuls ont survécu, contre 51 % de ceux qui voyageaient accompagnés.